# 26_E18 - UI Frontend para worklist y reporte IA

Este notebook genera un scaffold de frontend para visualizar la salida del agente IA ya validada en E14/E16/E17.

No entrena modelos. No modifica inferencia. Genera una pantalla MVP para:

- resumen del agente,
- distribución de prioridades,
- worklist de casos,
- flags de calidad,
- confianza,
- acción recomendada,
- payload listo para consumir desde Spring Boot.


In [ ]:
# Dependencias estándar
import json
import shutil
from pathlib import Path
from datetime import datetime

import pandas as pd

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
REPO_ROOT = PFI_ROOT / "repo"

E16_ROOT = PFI_ROOT / "results" / "E16_backend_integration_smoke"
E17_ROOT = PFI_ROOT / "results" / "E17_spring_boot_bridge_scaffold"
E18_ROOT = PFI_ROOT / "results" / "E18_frontend_worklist_ui"

DOCS_ROOT = PFI_ROOT / "docs"
FRONTEND_ROOT = REPO_ROOT / "frontend_ai_worklist"
SRC_ROOT = FRONTEND_ROOT / "src"
MOCK_ROOT = SRC_ROOT / "mock"

E16_SAMPLE_PATH = E16_ROOT / "E16_agent_report_response_sample.json"
E16_REPORT_PATH = E16_ROOT / "E16_backend_integration_report.json"
E17_REPORT_PATH = E17_ROOT / "E17_bridge_scaffold_report.json"

for p in [E18_ROOT, DOCS_ROOT, FRONTEND_ROOT, SRC_ROOT, MOCK_ROOT, REPO_ROOT / "backlogProducto"]:
    p.mkdir(parents=True, exist_ok=True)

print("PFI_ROOT:", PFI_ROOT)
print("REPO_ROOT:", REPO_ROOT, "| exists:", REPO_ROOT.exists())
print("FRONTEND_ROOT:", FRONTEND_ROOT)
print("E16_SAMPLE_PATH:", E16_SAMPLE_PATH, "| exists:", E16_SAMPLE_PATH.exists())
print("E16_REPORT_PATH:", E16_REPORT_PATH, "| exists:", E16_REPORT_PATH.exists())
print("E17_REPORT_PATH:", E17_REPORT_PATH, "| exists:", E17_REPORT_PATH.exists())

assert REPO_ROOT.exists(), f"No existe REPO_ROOT: {REPO_ROOT}"
assert E16_SAMPLE_PATH.exists(), f"No existe sample E16: {E16_SAMPLE_PATH}"


## 1. Carga del contrato validado en E16

Usamos el sample real de `/agent/report`, generado desde el servicio FastAPI y el bridge E17.


In [ ]:
agent_report = json.loads(E16_SAMPLE_PATH.read_text(encoding="utf-8"))
e16_report = json.loads(E16_REPORT_PATH.read_text(encoding="utf-8")) if E16_REPORT_PATH.exists() else {}
e17_report = json.loads(E17_REPORT_PATH.read_text(encoding="utf-8")) if E17_REPORT_PATH.exists() else {}

summary = agent_report.get("summary", {})
items = agent_report.get("items", [])

print("Sample keys:", list(agent_report.keys()))
print("Items:", len(items))
print("Summary:", json.dumps(summary, indent=2, ensure_ascii=False))

assert "summary" in agent_report
assert "items" in agent_report
assert len(items) > 0


## 2. Generación del frontend React/Vite

Se genera un frontend autocontenido. En producción consumiría el backend Spring Boot en:

```text
/api/ai/agent/report
```

Si el endpoint no responde, usa el mock real generado en E16.


In [ ]:
def write_text(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.strip() + "\n", encoding="utf-8")
    print("Wrote:", path)


package_json = {
    "name": "pfi-ai-worklist-ui",
    "version": "0.1.0",
    "private": True,
    "type": "module",
    "scripts": {
        "dev": "vite --host 0.0.0.0 --port 5173",
        "build": "vite build",
        "preview": "vite preview --host 0.0.0.0 --port 5173"
    },
    "dependencies": {
        "@vitejs/plugin-react": "^4.3.0",
        "vite": "^5.4.0",
        "typescript": "^5.5.0",
        "react": "^18.3.1",
        "react-dom": "^18.3.1"
    },
    "devDependencies": {}
}

write_text(FRONTEND_ROOT / "package.json", json.dumps(package_json, indent=2, ensure_ascii=False))

write_text(FRONTEND_ROOT / "index.html", """
<!doctype html>
<html lang="es">
  <head>
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <title>PFI AI Worklist</title>
  </head>
  <body>
    <div id="root"></div>
    <script type="module" src="/src/main.tsx"></script>
  </body>
</html>
""")

write_text(FRONTEND_ROOT / "vite.config.ts", """
import { defineConfig } from 'vite';
import react from '@vitejs/plugin-react';

export default defineConfig({
  plugins: [react()],
  server: {
    port: 5173,
    host: '0.0.0.0',
  },
});
""")

write_text(FRONTEND_ROOT / "tsconfig.json", """
{
  "compilerOptions": {
    "target": "ES2020",
    "useDefineForClassFields": true,
    "lib": ["DOM", "DOM.Iterable", "ES2020"],
    "allowJs": false,
    "skipLibCheck": true,
    "esModuleInterop": true,
    "allowSyntheticDefaultImports": true,
    "strict": false,
    "forceConsistentCasingInFileNames": true,
    "module": "ESNext",
    "moduleResolution": "Node",
    "resolveJsonModule": true,
    "isolatedModules": true,
    "noEmit": true,
    "jsx": "react-jsx"
  },
  "include": ["src"],
  "references": []
}
""")

write_text(SRC_ROOT / "types.ts", """
export type Priority = 'baja' | 'media' | 'alta' | string;

export interface AgentSummary {
  total_items: number;
  plane_distribution: Record<string, number>;
  priority_distribution: Record<string, number>;
  status_distribution: Record<string, number>;
  mean_fg_confidence?: number;
  mean_dice_macro_useful_classes?: number;
}

export interface AgentItem {
  agent_item_id: string;
  plane: string;
  model_key: string;
  case_ref: string;
  figure_path?: string | null;
  foreground_ratio?: number | null;
  n_components?: number | null;
  present_classes?: number[] | string | null;
  flags?: string[] | string | null;
  mean_confidence?: number | null;
  mean_fg_confidence?: number | null;
  agent_status: string;
  review_priority: Priority;
  agent_reasons?: string[] | string | null;
  recommended_action: string;
  human_review_required: boolean;
  dice_macro_useful_classes?: number | null;
  iou_macro_useful_classes?: number | null;
}

export interface AgentReportResponse {
  summary: AgentSummary;
  markdown?: string;
  items: AgentItem[];
}
""")

write_text(SRC_ROOT / "api.ts", """
import type { AgentReportResponse } from './types';
import mockReport from './mock/agentReport.sample.json';

const API_URL =
  import.meta.env.VITE_AI_AGENT_REPORT_URL ||
  '/api/ai/agent/report';

export async function fetchAgentReport(): Promise<AgentReportResponse> {
  try {
    const response = await fetch(API_URL);
    if (!response.ok) {
      throw new Error(`HTTP ${response.status}`);
    }
    return await response.json();
  } catch (error) {
    console.warn('Usando mock local porque no respondió el backend:', error);
    return mockReport as AgentReportResponse;
  }
}
""")

write_text(SRC_ROOT / "main.tsx", """
import React from 'react';
import ReactDOM from 'react-dom/client';
import App from './App';
import './styles.css';

ReactDOM.createRoot(document.getElementById('root')!).render(
  <React.StrictMode>
    <App />
  </React.StrictMode>
);
""")

write_text(SRC_ROOT / "App.tsx", r"""
import { useEffect, useMemo, useState } from 'react';
import { fetchAgentReport } from './api';
import type { AgentItem, AgentReportResponse } from './types';

function asList(value: unknown): string[] {
  if (!value) return [];
  if (Array.isArray(value)) return value.map(String);
  if (typeof value === 'string') {
    const trimmed = value.trim();
    if (!trimmed || trimmed === '[]') return [];
    return trimmed
      .replace(/^\[/, '')
      .replace(/\]$/, '')
      .split(',')
      .map((x) => x.replaceAll("'", '').replaceAll('"', '').trim())
      .filter(Boolean);
  }
  return [String(value)];
}

function fmt(value?: number | null, digits = 3): string {
  if (value === null || value === undefined || Number.isNaN(Number(value))) {
    return '-';
  }
  return Number(value).toFixed(digits);
}

function priorityClass(priority: string): string {
  if (priority === 'alta') return 'priority priority-high';
  if (priority === 'media') return 'priority priority-medium';
  return 'priority priority-low';
}

function StatusCard({ title, value, subtitle }: { title: string; value: string | number; subtitle?: string }) {
  return (
    <div className="card">
      <span className="card-title">{title}</span>
      <strong className="card-value">{value}</strong>
      {subtitle ? <span className="card-subtitle">{subtitle}</span> : null}
    </div>
  );
}

function ItemDetail({ item }: { item: AgentItem | null }) {
  if (!item) {
    return <aside className="detail empty">Seleccioná un caso para ver el detalle.</aside>;
  }

  const flags = asList(item.flags);
  const reasons = asList(item.agent_reasons);

  return (
    <aside className="detail">
      <div className="detail-header">
        <h2>{item.agent_item_id}</h2>
        <span className={priorityClass(item.review_priority)}>{item.review_priority}</span>
      </div>

      <p className="muted">{item.case_ref}</p>

      <div className="detail-grid">
        <div><b>Plano</b><span>{item.plane}</span></div>
        <div><b>Modelo</b><span>{item.model_key}</span></div>
        <div><b>Estado</b><span>{item.agent_status}</span></div>
        <div><b>Human review</b><span>{item.human_review_required ? 'Sí' : 'No'}</span></div>
        <div><b>Confianza FG</b><span>{fmt(item.mean_fg_confidence)}</span></div>
        <div><b>Dice útil</b><span>{fmt(item.dice_macro_useful_classes)}</span></div>
      </div>

      <h3>Acción recomendada</h3>
      <p>{item.recommended_action}</p>

      <h3>Flags</h3>
      {flags.length ? (
        <div className="tags">{flags.map((flag) => <span key={flag}>{flag}</span>)}</div>
      ) : (
        <p className="muted">Sin flags.</p>
      )}

      <h3>Razones del agente</h3>
      {reasons.length ? (
        <ul>{reasons.map((reason) => <li key={reason}>{reason}</li>)}</ul>
      ) : (
        <p className="muted">Sin razones adicionales.</p>
      )}

      {item.figure_path ? (
        <>
          <h3>Overlay</h3>
          <code>{item.figure_path}</code>
        </>
      ) : null}
    </aside>
  );
}

export default function App() {
  const [data, setData] = useState<AgentReportResponse | null>(null);
  const [selectedId, setSelectedId] = useState<string | null>(null);
  const [priorityFilter, setPriorityFilter] = useState<string>('todas');
  const [planeFilter, setPlaneFilter] = useState<string>('todos');

  useEffect(() => {
    fetchAgentReport().then((report) => {
      setData(report);
      setSelectedId(report.items?.[0]?.agent_item_id ?? null);
    });
  }, []);

  const items = data?.items ?? [];

  const filteredItems = useMemo(() => {
    return items.filter((item) => {
      const priorityOk = priorityFilter === 'todas' || item.review_priority === priorityFilter;
      const planeOk = planeFilter === 'todos' || item.plane === planeFilter;
      return priorityOk && planeOk;
    });
  }, [items, priorityFilter, planeFilter]);

  const selectedItem = items.find((item) => item.agent_item_id === selectedId) ?? filteredItems[0] ?? null;

  if (!data) {
    return <main className="app"><p>Cargando reporte IA...</p></main>;
  }

  const summary = data.summary;

  return (
    <main className="app">
      <header className="hero">
        <div>
          <p className="eyebrow">PFI · RM Lumbar · IA asistida</p>
          <h1>Worklist de revisión del agente IA</h1>
          <p>
            Pantalla MVP para visualizar resultados de segmentación, quality flags y prioridades.
            Todo resultado requiere revisión profesional.
          </p>
        </div>
      </header>

      <section className="cards">
        <StatusCard title="Total casos" value={summary.total_items} />
        <StatusCard title="Prioridad alta" value={summary.priority_distribution?.alta ?? 0} subtitle="Revisión prioritaria" />
        <StatusCard title="Prioridad media" value={summary.priority_distribution?.media ?? 0} subtitle="Revisar con atención" />
        <StatusCard title="Confianza FG media" value={fmt(summary.mean_fg_confidence)} />
        <StatusCard title="Dice útil medio" value={fmt(summary.mean_dice_macro_useful_classes)} />
      </section>

      <section className="filters">
        <label>
          Prioridad
          <select value={priorityFilter} onChange={(e) => setPriorityFilter(e.target.value)}>
            <option value="todas">Todas</option>
            <option value="alta">Alta</option>
            <option value="media">Media</option>
            <option value="baja">Baja</option>
          </select>
        </label>

        <label>
          Plano
          <select value={planeFilter} onChange={(e) => setPlaneFilter(e.target.value)}>
            <option value="todos">Todos</option>
            <option value="axial">Axial</option>
            <option value="sagittal">Sagital</option>
          </select>
        </label>
      </section>

      <section className="layout">
        <div className="table-wrap">
          <table>
            <thead>
              <tr>
                <th>ID</th>
                <th>Plano</th>
                <th>Caso</th>
                <th>Prioridad</th>
                <th>Estado</th>
                <th>Conf. FG</th>
                <th>Dice útil</th>
                <th>Flags</th>
              </tr>
            </thead>
            <tbody>
              {filteredItems.map((item) => (
                <tr
                  key={item.agent_item_id}
                  className={selectedItem?.agent_item_id === item.agent_item_id ? 'selected' : ''}
                  onClick={() => setSelectedId(item.agent_item_id)}
                >
                  <td>{item.agent_item_id}</td>
                  <td>{item.plane}</td>
                  <td>{item.case_ref}</td>
                  <td><span className={priorityClass(item.review_priority)}>{item.review_priority}</span></td>
                  <td>{item.agent_status}</td>
                  <td>{fmt(item.mean_fg_confidence)}</td>
                  <td>{fmt(item.dice_macro_useful_classes)}</td>
                  <td>{asList(item.flags).length || '-'}</td>
                </tr>
              ))}
            </tbody>
          </table>
        </div>

        <ItemDetail item={selectedItem} />
      </section>
    </main>
  );
}
""")

write_text(SRC_ROOT / "styles.css", r"""
:root {
  font-family: Inter, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
  color: #172033;
  background: #f5f7fb;
}

body {
  margin: 0;
}

.app {
  padding: 32px;
  max-width: 1440px;
  margin: 0 auto;
}

.hero {
  background: linear-gradient(135deg, #14213d, #284b8f);
  color: white;
  border-radius: 24px;
  padding: 32px;
  margin-bottom: 24px;
  box-shadow: 0 16px 40px rgba(20, 33, 61, 0.24);
}

.hero h1 {
  margin: 8px 0;
  font-size: 38px;
}

.eyebrow {
  text-transform: uppercase;
  letter-spacing: 0.12em;
  opacity: 0.8;
  font-size: 12px;
}

.cards {
  display: grid;
  grid-template-columns: repeat(5, minmax(160px, 1fr));
  gap: 16px;
  margin-bottom: 20px;
}

.card {
  background: white;
  padding: 18px;
  border-radius: 18px;
  box-shadow: 0 8px 24px rgba(23, 32, 51, 0.08);
}

.card-title,
.card-subtitle {
  display: block;
  color: #687087;
  font-size: 13px;
}

.card-value {
  display: block;
  font-size: 28px;
  margin: 8px 0;
}

.filters {
  display: flex;
  gap: 16px;
  margin-bottom: 20px;
}

.filters label {
  display: flex;
  flex-direction: column;
  gap: 6px;
  font-weight: 600;
}

select {
  border: 1px solid #d7dce8;
  border-radius: 12px;
  padding: 10px 12px;
  background: white;
}

.layout {
  display: grid;
  grid-template-columns: 1fr 360px;
  gap: 20px;
}

.table-wrap {
  overflow: auto;
  background: white;
  border-radius: 20px;
  box-shadow: 0 8px 24px rgba(23, 32, 51, 0.08);
}

table {
  width: 100%;
  border-collapse: collapse;
  min-width: 920px;
}

th,
td {
  padding: 14px 16px;
  border-bottom: 1px solid #edf0f6;
  text-align: left;
  font-size: 14px;
}

th {
  color: #687087;
  font-weight: 700;
  background: #fbfcff;
}

tr {
  cursor: pointer;
}

tr:hover,
tr.selected {
  background: #f3f7ff;
}

.priority {
  display: inline-flex;
  border-radius: 999px;
  padding: 4px 10px;
  font-weight: 700;
  font-size: 12px;
  text-transform: uppercase;
}

.priority-high {
  background: #ffe2e2;
  color: #a31313;
}

.priority-medium {
  background: #fff1cc;
  color: #8a5a00;
}

.priority-low {
  background: #e3f8e8;
  color: #146c2e;
}

.detail {
  background: white;
  border-radius: 20px;
  padding: 22px;
  box-shadow: 0 8px 24px rgba(23, 32, 51, 0.08);
  min-height: 480px;
}

.detail.empty {
  color: #687087;
}

.detail-header {
  display: flex;
  justify-content: space-between;
  align-items: center;
}

.detail h2 {
  margin: 0;
}

.detail h3 {
  margin-top: 22px;
  margin-bottom: 8px;
}

.muted {
  color: #687087;
}

.detail-grid {
  display: grid;
  grid-template-columns: 1fr 1fr;
  gap: 12px;
  margin: 16px 0;
}

.detail-grid div {
  background: #f7f9fd;
  padding: 12px;
  border-radius: 12px;
}

.detail-grid b,
.detail-grid span {
  display: block;
}

.detail-grid b {
  color: #687087;
  font-size: 12px;
}

.tags {
  display: flex;
  flex-wrap: wrap;
  gap: 8px;
}

.tags span {
  background: #eef2ff;
  color: #284b8f;
  padding: 6px 10px;
  border-radius: 999px;
  font-size: 12px;
}

code {
  display: block;
  white-space: pre-wrap;
  word-break: break-word;
  background: #f7f9fd;
  padding: 10px;
  border-radius: 10px;
  font-size: 12px;
}

@media (max-width: 1100px) {
  .cards {
    grid-template-columns: repeat(2, 1fr);
  }

  .layout {
    grid-template-columns: 1fr;
  }
}
""")

# Mock payload real de E16
write_text(MOCK_ROOT / "agentReport.sample.json", json.dumps(agent_report, indent=2, ensure_ascii=False))

write_text(FRONTEND_ROOT / "README.md", """
# E18 - Frontend AI Worklist UI

Scaffold React/Vite para visualizar la salida del agente IA.

## Objetivo

Mostrar en una pantalla MVP:

- resumen global del agente,
- distribución de prioridades,
- worklist de casos,
- flags de calidad,
- confianza,
- métricas útiles,
- acción recomendada,
- detalle por caso.

## Backend esperado

Por defecto intenta consumir:

```text
/api/ai/agent/report
```

Si el endpoint no responde, usa el mock local:

```text
src/mock/agentReport.sample.json
```

## Ejecutar

```bash
cd frontend_ai_worklist
npm install
npm run dev
```

## Integración

En producción, Spring Boot debe exponer `/api/ai/agent/report` y reenviar la respuesta del servicio Python FastAPI.
""")

write_text(FRONTEND_ROOT / ".env.example", """
VITE_AI_AGENT_REPORT_URL=/api/ai/agent/report
""")


## 3. Vista estática HTML para revisión rápida

Además del scaffold React, se genera una vista HTML estática para abrirla directamente desde Drive y sacar captura si hace falta.


In [ ]:
def priority_badge(priority: str) -> str:
    cls = {"alta": "high", "media": "medium", "baja": "low"}.get(priority, "low")
    return f'<span class="badge {cls}">{priority}</span>'

rows_html = []
for item in items:
    rows_html.append(f"""
    <tr>
      <td>{item.get('agent_item_id')}</td>
      <td>{item.get('plane')}</td>
      <td>{item.get('case_ref')}</td>
      <td>{priority_badge(str(item.get('review_priority')))}</td>
      <td>{item.get('agent_status')}</td>
      <td>{item.get('mean_fg_confidence', '-')}</td>
      <td>{item.get('dice_macro_useful_classes', '-')}</td>
      <td>{item.get('recommended_action')}</td>
    </tr>
    """)

preview_html = f"""
<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<title>E18 Preview - AI Worklist</title>
<style>
body {{ font-family: Inter, Arial, sans-serif; background:#f5f7fb; color:#172033; margin:0; padding:32px; }}
.hero {{ background:linear-gradient(135deg,#14213d,#284b8f); color:white; padding:28px; border-radius:24px; margin-bottom:20px; }}
.cards {{ display:grid; grid-template-columns:repeat(5,1fr); gap:14px; margin-bottom:20px; }}
.card {{ background:white; border-radius:16px; padding:16px; box-shadow:0 8px 22px rgba(20,33,61,.08); }}
.card b {{ font-size:28px; display:block; margin-top:6px; }}
table {{ width:100%; border-collapse:collapse; background:white; border-radius:18px; overflow:hidden; box-shadow:0 8px 22px rgba(20,33,61,.08); }}
th, td {{ padding:12px 14px; border-bottom:1px solid #edf0f6; text-align:left; font-size:14px; }}
th {{ background:#fbfcff; color:#687087; }}
.badge {{ border-radius:999px; padding:4px 10px; font-weight:700; font-size:12px; text-transform:uppercase; }}
.high {{ background:#ffe2e2; color:#a31313; }}
.medium {{ background:#fff1cc; color:#8a5a00; }}
.low {{ background:#e3f8e8; color:#146c2e; }}
</style>
</head>
<body>
<section class="hero">
  <p>PFI · RM Lumbar · IA asistida</p>
  <h1>Worklist de revisión del agente IA</h1>
  <p>Vista estática generada en E18 para validar layout y contrato de datos.</p>
</section>

<section class="cards">
  <div class="card">Total casos <b>{summary.get('total_items')}</b></div>
  <div class="card">Alta <b>{summary.get('priority_distribution', {}).get('alta', 0)}</b></div>
  <div class="card">Media <b>{summary.get('priority_distribution', {}).get('media', 0)}</b></div>
  <div class="card">Baja <b>{summary.get('priority_distribution', {}).get('baja', 0)}</b></div>
  <div class="card">Dice útil medio <b>{summary.get('mean_dice_macro_useful_classes', 0):.3f}</b></div>
</section>

<table>
<thead>
<tr>
  <th>ID</th>
  <th>Plano</th>
  <th>Caso</th>
  <th>Prioridad</th>
  <th>Estado</th>
  <th>Conf. FG</th>
  <th>Dice útil</th>
  <th>Acción recomendada</th>
</tr>
</thead>
<tbody>
{''.join(rows_html)}
</tbody>
</table>
</body>
</html>
"""

preview_path = DOCS_ROOT / "E18_frontend_static_preview.html"
write_text(preview_path, preview_html)
print("Preview HTML:", preview_path)


## 4. Documentación E18

Se generan contratos y prompt para Codex/frontend.


In [ ]:
frontend_contract_md = f"""
# E18 - Contrato Frontend Worklist IA

## Endpoint esperado

```text
GET /api/ai/agent/report
```

Este endpoint debe ser expuesto por Spring Boot y puede reenviar la respuesta del servicio Python.

## Summary

```json
{json.dumps(summary, indent=2, ensure_ascii=False)}
```

## Campos mínimos por item

```text
agent_item_id
plane
model_key
case_ref
figure_path
foreground_ratio
n_components
present_classes
flags
mean_confidence
mean_fg_confidence
agent_status
review_priority
agent_reasons
recommended_action
human_review_required
dice_macro_useful_classes
iou_macro_useful_classes
```

## Uso en UI

La pantalla E18 debe mostrar:

- tarjetas de resumen,
- filtros por plano y prioridad,
- tabla de worklist,
- detalle por caso,
- flags y razones del agente,
- acción recomendada,
- confirmación de revisión humana obligatoria.
"""

codex_prompt_md = """
# Prompt para Codex - E18 Frontend AI Worklist

Necesito integrar el scaffold `frontend_ai_worklist` al frontend principal del proyecto.

Contexto:
- El backend Spring Boot expondrá `GET /api/ai/agent/report`.
- Ese endpoint devuelve `{ summary, markdown, items }`.
- `items` contiene casos axiales y sagitales con prioridad, flags, confianza, métricas y acción recomendada.
- La revisión humana es obligatoria.

Tareas:
1. Revisar `frontend_ai_worklist/src/App.tsx`.
2. Adaptarlo al stack real del frontend si existe.
3. Mantener filtros por plano y prioridad.
4. Mantener vista de detalle por caso.
5. Mostrar de forma clara prioridad alta/media/baja.
6. No presentar el resultado como diagnóstico automático.
7. Agregar manejo de loading/error.
8. Preparar llamada al backend `/api/ai/agent/report`.
9. Dejar componentes reutilizables si corresponde.

Criterio de aceptación:
- La pantalla muestra total de casos, prioridades, lista y detalle.
- Funciona con mock local si el backend no responde.
- El texto indica que el resultado requiere revisión profesional.
"""

conclusion_md = f"""
# E18 - Frontend Worklist UI

Estado: scaffold generado.

## Resultado

Se creó un frontend React/Vite para visualizar la worklist del agente IA y el reporte de revisión profesional.

## Fuente de datos

- E16 sample: `{E16_SAMPLE_PATH}`
- Endpoint esperado: `/api/ai/agent/report`

## Resumen cargado

- total_items: {summary.get('total_items')}
- priority_distribution: {summary.get('priority_distribution')}
- plane_distribution: {summary.get('plane_distribution')}
- mean_fg_confidence: {summary.get('mean_fg_confidence')}
- mean_dice_macro_useful_classes: {summary.get('mean_dice_macro_useful_classes')}

## Archivos principales

- `frontend_ai_worklist/src/App.tsx`
- `frontend_ai_worklist/src/api.ts`
- `frontend_ai_worklist/src/types.ts`
- `frontend_ai_worklist/src/mock/agentReport.sample.json`
- `docs/E18_frontend_static_preview.html`

## Decisión

E18 deja lista una pantalla MVP para visualizar la cadena IA → agente → revisión humana.
"""

e18_plan_md = """
# E18 - UI frontend para worklist IA

Estado: En progreso.

Objetivo: crear una pantalla MVP para visualizar el reporte del agente IA.

Alcance:
- Scaffold React/Vite.
- Mock real desde E16.
- Tabla de worklist.
- Filtros por plano y prioridad.
- Detalle por caso.
- Contrato frontend/backend.
- Vista HTML estática para revisión rápida.

Decisión: el frontend consume Spring Boot, no consume directamente el servicio Python.
"""

write_text(DOCS_ROOT / "E18_frontend_worklist_contract.md", frontend_contract_md)
write_text(DOCS_ROOT / "E18_codex_frontend_prompt.md", codex_prompt_md)
write_text(DOCS_ROOT / "E18_frontend_ui_conclusion.md", conclusion_md)
write_text(REPO_ROOT / "backlogProducto" / "E18_plan.md", e18_plan_md)


## 5. Inventario y checks estáticos

In [ ]:
generated_paths = sorted([p for p in FRONTEND_ROOT.rglob("*") if p.is_file()])
inventory_rows = []
for p in generated_paths:
    inventory_rows.append({
        "relative_path": str(p.relative_to(REPO_ROOT)),
        "size_bytes": p.stat().st_size,
        "suffix": p.suffix,
    })

inventory_df = pd.DataFrame(inventory_rows)
inventory_csv = E18_ROOT / "E18_frontend_file_inventory.csv"
inventory_df.to_csv(inventory_csv, index=False)

display(inventory_df)
print("Inventory:", inventory_csv)

required_files = [
    FRONTEND_ROOT / "README.md",
    FRONTEND_ROOT / "package.json",
    FRONTEND_ROOT / "index.html",
    FRONTEND_ROOT / "vite.config.ts",
    FRONTEND_ROOT / "tsconfig.json",
    FRONTEND_ROOT / ".env.example",
    SRC_ROOT / "main.tsx",
    SRC_ROOT / "App.tsx",
    SRC_ROOT / "api.ts",
    SRC_ROOT / "types.ts",
    SRC_ROOT / "styles.css",
    MOCK_ROOT / "agentReport.sample.json",
]

required_summary_keys = ["total_items", "plane_distribution", "priority_distribution", "status_distribution"]
required_item_keys = [
    "agent_item_id", "plane", "model_key", "case_ref", "figure_path",
    "foreground_ratio", "n_components", "present_classes", "flags",
    "mean_confidence", "mean_fg_confidence", "agent_status",
    "review_priority", "agent_reasons", "recommended_action",
    "human_review_required"
]

checks = []

for path in required_files:
    checks.append({
        "check": "file_exists",
        "target": str(path.relative_to(REPO_ROOT)),
        "ok": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else None,
    })

for key in required_summary_keys:
    checks.append({
        "check": "summary_key",
        "target": key,
        "ok": key in summary,
        "size_bytes": None,
    })

first_item = items[0] if items else {}
for key in required_item_keys:
    checks.append({
        "check": "item_key",
        "target": key,
        "ok": key in first_item,
        "size_bytes": None,
    })

checks_df = pd.DataFrame(checks)
checks_csv = E18_ROOT / "E18_frontend_static_checks.csv"
checks_df.to_csv(checks_csv, index=False)

display(checks_df)
print("Checks:", checks_csv)
print("All OK:", bool(checks_df["ok"].all()))


## 6. Reporte final E18

In [ ]:
report = {
    "notebook": "26_E18_frontend_worklist_ui_scaffold",
    "goal": "generate frontend UI scaffold for AI agent worklist and report",
    "frontend_root": str(FRONTEND_ROOT),
    "source_e16_sample": str(E16_SAMPLE_PATH),
    "outputs": {
        "inventory_csv": str(inventory_csv),
        "static_checks_csv": str(checks_csv),
        "static_preview_html": str(preview_path),
        "frontend_contract_md": str(DOCS_ROOT / "E18_frontend_worklist_contract.md"),
        "codex_prompt_md": str(DOCS_ROOT / "E18_codex_frontend_prompt.md"),
        "conclusion_md": str(DOCS_ROOT / "E18_frontend_ui_conclusion.md"),
    },
    "summary": {
        "generated_files": int(len(inventory_df)),
        "required_files": int(len(required_files)),
        "all_static_checks_ok": bool(checks_df["ok"].all()),
        "e16_total_items": int(summary.get("total_items", 0)),
        "e16_priority_distribution": summary.get("priority_distribution", {}),
        "e16_plane_distribution": summary.get("plane_distribution", {}),
    },
    "architecture_decision": {
        "frontend_consumes_spring_boot": True,
        "spring_boot_consumes_python_service": True,
        "python_service_runs_ai_agent": True,
        "human_review_required": True,
    },
    "decision": "frontend_worklist_ui_scaffold_ready_for_mvp_integration",
}

report_path = E18_ROOT / "E18_frontend_ui_report.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

print("Reporte E18:", report_path)
print(json.dumps(report, indent=2, ensure_ascii=False))


## 7. Commit sugerido

Después de revisar los outputs, guardar el notebook y correr:

```bash
%cd /content/drive/MyDrive/PFI_MVP/repo
!git status
!git add frontend_ai_worklist/ backlogProducto/E18_plan.md notebooks/26_E18_frontend_worklist_ui_scaffold.ipynb
!git commit -m "Add E18 frontend AI worklist UI scaffold"
!git push
```

Si se quieren versionar los docs generados fuera de `/repo`, copiarlos previamente dentro del repositorio.
